In [0]:
from pyspark.sql.functions import col, lit, sha2, concat_ws, current_timestamp, expr
from delta.tables import DeltaTable

test_series = "EMM_EPMPU_PTE_NUS_DPG"

In [0]:
spark.table("dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_prices") \
    .filter(col("series_bk") == test_series) \
    .orderBy("effective_from") \
    .show(truncate=False)

In [0]:
raw_simulated_df = (spark.table("dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_prices")
    .filter(col("series_bk") == test_series)
    .filter(col("is_current") == True)
    .drop("_ingested_at")
    .withColumn("price", col("price") + 10)
    .withColumn("effective_from", expr("date_add(effective_from, 7)"))
    .withColumn("price_sk", sha2(concat_ws("||", col("series_bk"), col("effective_from").cast("string")), 256))
    .withColumn("effective_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("_ingested_at", current_timestamp())
    .select("price_sk", "series_bk", "product_name", "units", "price",
            "effective_from", "effective_to", "is_current", "_source_system", "_ingested_at")
)

rows = raw_simulated_df.collect()
simulated_update_df = spark.createDataFrame(rows, schema=raw_simulated_df.schema)

simulated_update_df.show(truncate=False)

In [0]:
prices_target = DeltaTable.forName(
    spark, "dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_prices"
)

result_step1 = (prices_target.alias("t")
    .merge(simulated_update_df.alias("s"), "t.series_bk = s.series_bk AND t.is_current = true")
    .whenMatchedUpdate(
        condition="t.price <> s.price",
        set={"effective_to": "s.effective_from", "is_current": "false"}
    )
    .execute()
)
display(result_step1)

In [0]:
result_step2 = (prices_target.alias("t")
    .merge(
        simulated_update_df.alias("s"),
        "t.series_bk = s.series_bk AND t.effective_from = s.effective_from"
    )
    .whenNotMatchedInsert(
        values={
            "price_sk": "s.price_sk",
            "series_bk": "s.series_bk",
            "product_name": "s.product_name",
            "units": "s.units",
            "price": "s.price",
            "effective_from": "s.effective_from",
            "effective_to": "s.effective_to",
            "is_current": "s.is_current",
            "_source_system": "s._source_system",
            "_ingested_at": "s._ingested_at"
        }
    )
    .execute()
)
display(result_step2)

In [0]:
%sql
SELECT price, effective_from, effective_to, is_current
FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_prices
WHERE series_bk = 'EMM_EPMPU_PTE_NUS_DPG'
ORDER BY effective_from DESC
LIMIT 3

In [0]:
%sql
SELECT is_current, COUNT(*) 
FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_prices
GROUP BY is_current